# NER System — Exploration & Analysis
Use this notebook to explore the dataset and inspect model predictions interactively.

In [ ]:
from datasets import load_dataset
import pandas as pd
from collections import Counter

dataset = load_dataset('conll2003', trust_remote_code=True)
print(dataset)

In [ ]:
# Label distribution
LABELS = ['O','B-PER','I-PER','B-ORG','I-ORG','B-LOC','I-LOC','B-MISC','I-MISC']
counter = Counter()
for ex in dataset['train']:
    counter.update(LABELS[t] for t in ex['ner_tags'])

df = pd.DataFrame(counter.most_common(), columns=['Tag', 'Count'])
display(df)

In [ ]:
# Sample sentences
for ex in dataset['train'].select(range(3)):
    pairs = list(zip(ex['tokens'], [LABELS[t] for t in ex['ner_tags']]))
    print(pairs)
    print()

In [ ]:
# Training history (after training)
import json, matplotlib.pyplot as plt

with open('../models/training_history.json') as f:
    history = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'],   label='Val Loss')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(history['val_f1'], color='green', label='Val F1')
axes[1].set_title('Validation F1'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Live prediction via API
import requests

text = 'Elon Musk founded SpaceX in Hawthorne, California.'
r = requests.post('http://localhost:8000/predict', json={'text': text})
data = r.json()

for tt in data['tokens']:
    if tt['tag'] != 'O':
        print(f"{tt['token']:20s} → {tt['tag']}")

print('\nEntities:')
for ent in data['entities']:
    print(f"  [{ent['label']}] {ent['text']}")